In [ ]:
from langchain.tools import tool
from langchain_openai import ChatOpenAI
from langchain.agents import AgentState, create_agent
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import SummarizationMiddleware

import os
from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

credential_key=os.getenv("credential_key")
send_system_name=os.getenv("send_system_name")
model=os.getenv("model")
api_base_url=os.getenv("api_base_url")
user_id=os.getenv("user_id")

os.environ["OPENAI_API_KEY"] = 'api_key'

model = ChatOpenAI(
    model=model,
    base_url=api_base_url,
    default_headers={
        'x-dep-ticket': credential_key,
        'Send-System-Name': send_system_name,
        'User-Id': user_id,
        'User-Type': "AD_ID",
        'Prompt-Msg-Id': str(uuid.uuid4()),
        'Completion-Msg-Id': str(uuid.uuid4())
    },
    temperature=0.7,
)

c:\Users\rando\anaconda3\envs\langchain\Lib\site-packages\langgraph\checkpoint\serde\encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


In [2]:
from langchain_core.runnables import RunnableConfig

@tool
def save_user_info(info: str) -> str:
    """사용자에 대한 정보를 기록합니다. 추후 대화에 도움이 될 만한 사용자 정보를 기록하는데 사용하세요."""
    
    with open(f"user_profile.txt", "w") as f:
        f.write(info)
    
    return "사용자 정보를 저장했습니다."

@tool
def load_user_info() -> str:
    """저장된 사용자 정보를 불러옵니다."""
    
    try:
        with open(f"user_profile.txt", "r") as f:
            return f.read()
    except FileNotFoundError:
        return "저장된 정보가 없습니다."

@tool
def calculator(a: int, b: int, operator: str) -> float:
    """
    2개의 정수 a, b를 입력 받으면 연산자(operator) string에 
    해당하는 연산을 수행하고 그 결과를 반환하는 함수.
    operator에는 'plus', 'minus', 'multiply', 'divide' 4가지만 들어갈 수 있다.
    """
    if operator == "plus":
        return a + b
    if operator == "minus":
        return a - b
    if operator == "multiply":
        return a * b
    if operator == "divide":
        return a / b

In [3]:
with open("agents.md", encoding="utf-8") as f:
    system_prompt = f.read()

agent = create_agent(
    model,
    tools=[save_user_info, load_user_info, calculator],
    checkpointer=InMemorySaver()
)

In [ ]:
# 1. 에이전트 스트리밍 실행
for chunk in agent.stream(
    {"messages": [{"role": "user", "content": "안녕, 내 이름은 민희야. 1+1은 뭐야? 그리고 그 이유가 뭐야?"}]},
    stream_mode="messages",
    config={"configurable": {"thread_id": "streaming_test"}}
):
    # chunk는 (메시지 내용, 메타데이터)로 구성된 튜플 형태입니다.
    message_chunk, metadata = chunk
    
    # 내용(content)이 있는 경우에만 출력합니다.
    # end=""와 flush=True를 써야 줄바꿈 없이 한 글자씩 실시간으로 이어져서 보입니다.
    if message_chunk.content:
        print(message_chunk.content, end="", flush=True)


안녕 민희야! 1+1은 **2**야.

이유는 아주 간단해:
- 사과 1개가 있고
- 또 사과 1개를 더하면
- 사과가 총 2개가 되기 때문이야.

즉, **1개 + 1개 = 2개**야.

--- 스트리밍 완료 ---
